# Baseline pipeline: CatBoost + региональный OOF-признак

Один простой, честный baseline (без ансамблей и тюнинга) + региональный
OOF-энкодинг `adminarea` взвешенной медианой. Метрика — **WMAE**, вес `w`
используется как `sample_weight`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold as _PlainKFold  # comparison only, see section 4

from src.config import (
    CSV_READ_KWARGS, TRAIN_PATH, TEST_PATH, TARGET_COL, WEIGHT_COL, ID_COL,
    DATE_COL, CATEGORICAL_COLS, REGION_COL, MIN_GROUP_COUNT, N_FOLDS,
    RANDOM_SEED, OUTPUTS_DIR, PARTIAL_OUTPUTS_DIR,
)
from src.preprocessing import preprocess
from src.metrics import wmae, weighted_median, bootstrap_wmae_ci
from src.validation import get_folds
from src.region_encoding import (
    oof_region_encoding, fit_full_region_encoding, apply_region_encoding,
    compute_region_stats, apply_region_stats,
)
from src.modeling import LogTargetCatBoost, prepare_cat_features

np.random.seed(RANDOM_SEED)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Препроцессинг

In [2]:
train_raw = pd.read_csv(TRAIN_PATH, **CSV_READ_KWARGS, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, **CSV_READ_KWARGS, low_memory=False)

train, train_fixed_cols = preprocess(train_raw, is_train=True)
test, test_fixed_cols = preprocess(test_raw, is_train=False)

print("train:", train.shape, "| test:", test.shape)
print(f"dtype-fixed columns: {len(train_fixed_cols)} (train), {len(test_fixed_cols)} (test)")


train: (76786, 224) | test: (73214, 222)
dtype-fixed columns: 33 (train), 33 (test)


In [3]:
RAW_FEATURE_COLS = [c for c in train.columns if c not in [ID_COL, TARGET_COL, WEIGHT_COL]]
CAT_COLS = list(CATEGORICAL_COLS)  # includes 'dt'

y = train[TARGET_COL].to_numpy()
w = train[WEIGHT_COL].to_numpy()

print(f"raw feature columns: {len(RAW_FEATURE_COLS)}")
print(f"categorical columns (native CatBoost support): {CAT_COLS}")


raw feature columns: 221
categorical columns (native CatBoost support): ['dt', 'gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref', 'dp_address_unique_regions', 'period_last_act_ad']


`dt` остаётся одной из 8 категориальных колонок для CatBoost (нативная
поддержка категорий, без ручного FE над датами — см. раздел 6 ТЗ, "не
делать сложный feature engineering"). Пропуски в категориальных колонках
заполняются меткой `'missing'` (`prepare_cat_features`), остальные 33
misclassified-числовые уже приведены к float в `preprocess()`.

## 2. Метрика

In [4]:
# Быстрая демонстрация метрики (полное покрытие - в tests/test_metrics.py)
demo_y_true = np.array([10.0, 20.0, 30.0])
demo_y_pred = np.array([12.0, 18.0, 35.0])
demo_w = np.array([1.0, 1.0, 3.0])
print("unweighted MAE:", wmae(demo_y_true, demo_y_pred))
print("WMAE (weighted):", wmae(demo_y_true, demo_y_pred, demo_w))


unweighted MAE: 3.0
WMAE (weighted): 3.8


## 3. Региональный признак (OOF weighted-median encoding)

In [5]:
# StratifiedKFold по децилям target (src/validation.py) - тот же сплит
# используется и для региональной энкодировки, и для CV модели ниже.
folds = get_folds(y, n_folds=N_FOLDS, seed=RANDOM_SEED)

# OOF-энкодинг считается один раз по этим же 5 фолдам, которыми ниже будет
# оцениваться и сама модель - поэтому кодировка каждой строки в fold i
# зависит только от строк ВНЕ fold i (см. src/region_encoding.py).
region_oof_train = oof_region_encoding(
    train, y, w, folds, region_col=REGION_COL, min_count=MIN_GROUP_COUNT
)
train["region_income_encoding"] = region_oof_train
print("region_income_encoding (train) - sample:")
train[[REGION_COL, "region_income_encoding"]].head()


region_income_encoding (train) - sample:


,adminarea,region_income_encoding
0,Свердловская область,64525.633269
1,Краснодарский край,60041.858100
2,Новосибирская область,60649.850000
3,Хабаровский край,67555.989019
4,Новосибирская область,58841.440800


## 4. Валидация

In [6]:
print(f"StratifiedKFold (по децилям target): {N_FOLDS} folds, shuffle=True, seed={RANDOM_SEED}")
for i, (tr_idx, val_idx) in enumerate(folds):
    print(f"  fold {i}: train={len(tr_idx)}, val={len(val_idx)}")


StratifiedKFold (по децилям target): 5 folds, shuffle=True, seed=42
  fold 0: train=61428, val=15358
  fold 1: train=61429, val=15357
  fold 2: train=61429, val=15357
  fold 3: train=61429, val=15357
  fold 4: train=61429, val=15357


**Почему StratifiedKFold, а не обычный `KFold`**: случайный `shuffle`-сплит
может по случайности отправить непропорциональную долю верхнего дециля
target (редкие, но самые дорогие по `w` строки — см. раздел 0.3/раздел 7
ноутбука 1) в один фолд. Стратификация по `pd.qcut(target, 10)` держит
распределение таргета (и, соответственно, хвост) сбалансированным между
фолдами — см. количественное сравнение в разделе 5 ниже.

In [7]:
# Раздел 4 ноутбука 1 показал: dt train (2024-01..06) и test (2024-07..11)
# не пересекаются, adversarial AUC~0.85 -> реальный temporal shift.
# Добавляем time-based split: последний месяц train как валидация.
print(train[DATE_COL].value_counts().sort_index())

time_cutoff = train[DATE_COL].max()  # 2024-06-30
time_train_mask = (train[DATE_COL] < time_cutoff).to_numpy()
time_val_mask = (train[DATE_COL] == time_cutoff).to_numpy()
print(f"time-based split: train={time_train_mask.sum()}, val={time_val_mask.sum()} (holdout month: {time_cutoff.date()})")


dt
2024-01-31     7243
2024-02-29     8865
2024-03-31    13413
2024-04-30    14858
2024-05-31    16193
2024-06-30    16214
Name: count, dtype: int64
time-based split: train=60572, val=16214 (holdout month: 2024-06-30)


## 5. Sanity floor: региональный тривиальный baseline

In [8]:
sanity_floor_wmae = wmae(y, region_oof_train, w)
print(f"Sanity floor (OOF region weighted-median baseline) WMAE: {sanity_floor_wmae:,.0f}")
print("Reference from ТЗ section 0.6 (best region baseline): ~120 225")


Sanity floor (OOF region weighted-median baseline) WMAE: 120,195
Reference from ТЗ section 0.6 (best region baseline): ~120 225


Ожидание из ТЗ (раздел 0.6) — около 120 000 — воспроизведено с точностью
до сотен единиц. Полная модель ниже должна **заметно** побить этот
ориентир, иначе остальные ~200 признаков не используются эффективно.

### Наивный std vs bootstrap CI (демонстрация на этом дешёвом baseline)

ТЗ v2 требует показать, что обычный `std` по 5 fold-level числам — не самая
точная оценка неопределённости WMAE: он оценивается по выборке размера
всего 5, тогда как bootstrap использует все ~77k OOF-предсказаний целиком.
Ниже — три варианта на одном и том же region-only baseline: (1) наивный
`std` на **старом** случайном `KFold` (до перехода на стратификацию),
(2) наивный `std` на новом `StratifiedKFold`, (3) bootstrap 95% CI на
пуле всех OOF-предсказаний.

In [9]:
# (1) наивный std на СТАРОМ случайном KFold (для сравнения, не используется дальше)
plain_folds = list(_PlainKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED).split(np.arange(len(train))))
region_oof_plain = oof_region_encoding(train, y, w, plain_folds, region_col=REGION_COL, min_count=MIN_GROUP_COUNT)
plain_fold_scores = [wmae(y[val], region_oof_plain[val], w[val]) for _, val in plain_folds]
naive_std_plain_kfold = np.std(plain_fold_scores)

# (2) наивный std на новом StratifiedKFold (folds, region_oof_train уже посчитаны выше)
strat_fold_scores = [wmae(y[val], region_oof_train[val], w[val]) for _, val in folds]
naive_std_stratified = np.std(strat_fold_scores)

# (3) bootstrap 95% CI на пуле всех OOF-предсказаний (стратифицированных)
sanity_floor_ci = bootstrap_wmae_ci(y, region_oof_train, w, n_boot=2000, seed=RANDOM_SEED)

print(f"(1) naive std, plain KFold (pre-stratification):   {naive_std_plain_kfold:,.0f}")
print(f"(2) naive std, StratifiedKFold:                     {naive_std_stratified:,.0f}")
print(f"(3) bootstrap std, pooled OOF (StratifiedKFold):     {sanity_floor_ci['std']:,.0f}")
print(f"    bootstrap 95% CI: [{sanity_floor_ci['ci_low']:,.0f}, {sanity_floor_ci['ci_high']:,.0f}]")
print("Reference from TZ v2 log (this baseline): naive std 3389 (plain KFold) -> 1297 (stratified) -> bootstrap std 1261")


(1) naive std, plain KFold (pre-stratification):   3,031
(2) naive std, StratifiedKFold:                     1,160
(3) bootstrap std, pooled OOF (StratifiedKFold):     1,260
    bootstrap 95% CI: [117,707, 122,633]
Reference from TZ v2 log (this baseline): naive std 3389 (plain KFold) -> 1297 (stratified) -> bootstrap std 1261


Стратификация сама по себе уже заметно сузила наивный `std` (тяжёлый
хвост target больше не концентрируется случайно в одном фолде), а bootstrap
CI на пуле OOF почти в точности совпадает со стратифицированным наивным
`std` — то есть после перехода на `StratifiedKFold` наивная оценка
перестаёт систематически завышать неопределённость. Именно поэтому раздел
6 ниже (модель) отчитывается по обеим метрикам: per-fold WMAE **и** pooled
OOF WMAE с bootstrap 95% CI.

## 6. Модель

In [10]:
def build_feature_matrix(df):
    X = df[RAW_FEATURE_COLS + ["region_income_encoding"]].copy()
    X[DATE_COL] = X[DATE_COL].astype(str)
    X = prepare_cat_features(X, CAT_COLS)
    return X

X_train_full = build_feature_matrix(train)
print("feature matrix shape:", X_train_full.shape)


feature matrix shape: (76786, 222)


In [11]:
fold_wmae = []
fold_best_iters = []
oof_model_preds = np.full(len(train), np.nan)

for i, (tr_idx, val_idx) in enumerate(folds):
    model = LogTargetCatBoost(cat_features=CAT_COLS)
    model.fit(
        X_train_full.iloc[tr_idx], y[tr_idx], sample_weight=w[tr_idx],
        eval_set=(X_train_full.iloc[val_idx], y[val_idx], w[val_idx]),
    )
    preds = model.predict(X_train_full.iloc[val_idx])
    oof_model_preds[val_idx] = preds

    fold_score = wmae(y[val_idx], preds, w[val_idx])
    fold_wmae.append(fold_score)
    fold_best_iters.append(model.model.get_best_iteration())
    print(f"fold {i}: WMAE={fold_score:,.0f}  (best_iteration={fold_best_iters[-1]})")

cv_wmae_mean = np.mean(fold_wmae)
cv_wmae_std = np.std(fold_wmae)
pooled_oof_wmae = wmae(y, oof_model_preds, w)
model_ci = bootstrap_wmae_ci(y, oof_model_preds, w, n_boot=2000, seed=RANDOM_SEED)

print(f"\nPer-fold CV WMAE: {cv_wmae_mean:,.0f} +/- {cv_wmae_std:,.0f}  (naive std over {N_FOLDS} folds)")
print(f"Pooled OOF WMAE:  {pooled_oof_wmae:,.0f}")
print(f"Pooled OOF bootstrap 95% CI: [{model_ci['ci_low']:,.0f}, {model_ci['ci_high']:,.0f}]  (bootstrap std={model_ci['std']:,.0f})")
print(f"Sanity floor (region-only):  {sanity_floor_wmae:,.0f}")
print(f"Improvement over sanity floor: {100 * (1 - cv_wmae_mean / sanity_floor_wmae):.1f}%")


fold 0: WMAE=72,259  (best_iteration=1997)


fold 1: WMAE=69,692  (best_iteration=1995)


fold 2: WMAE=71,665  (best_iteration=1992)


fold 3: WMAE=71,572  (best_iteration=1992)


fold 4: WMAE=69,803  (best_iteration=1999)



Per-fold CV WMAE: 70,998 +/- 1,049  (naive std over 5 folds)
Pooled OOF WMAE:  70,999
Pooled OOF bootstrap 95% CI: [69,254, 72,747]  (bootstrap std=888)
Sanity floor (region-only):  120,195
Improvement over sanity floor: 40.9%


**Наивный std vs bootstrap CI на самой модели**: как и в разделе 5, наивный
`std` по 5 fold-level числам оценивается на выборке размера 5, тогда как
bootstrap CI использует все ~77k pooled OOF-предсказаний. Ожидается, что
после перехода на `StratifiedKFold` оба согласуются гораздо теснее, чем на
старом случайном `KFold` (где наивный `std` был ощутимо шире bootstrap-оценки
неопределённости).

In [12]:
# Time-based validation (see section 4): region encoding refit on the
# earlier months only, to avoid leaking the held-out month into the encoding.
time_stats, time_fallback = compute_region_stats(
    train.loc[time_train_mask, REGION_COL].to_numpy(),
    y[time_train_mask], w[time_train_mask], min_count=MIN_GROUP_COUNT,
)
# stats were fit only on the pre-cutoff rows, so applying them to every
# row (train and held-out val) is leakage-safe by construction.
region_time_encoding = apply_region_stats(train[REGION_COL].to_numpy(), time_stats, time_fallback)
train_time = train.copy()
train_time["region_income_encoding"] = region_time_encoding
X_time = build_feature_matrix(train_time)

model_time = LogTargetCatBoost(cat_features=CAT_COLS)
model_time.fit(
    X_time.iloc[time_train_mask], y[time_train_mask], sample_weight=w[time_train_mask],
    eval_set=(X_time.iloc[time_val_mask], y[time_val_mask], w[time_val_mask]),
)
time_preds = model_time.predict(X_time.iloc[time_val_mask])
time_wmae = wmae(y[time_val_mask], time_preds, w[time_val_mask])

print(f"Time-based holdout WMAE (train<{time_cutoff.date()} -> val={time_cutoff.date()}): {time_wmae:,.0f}")
print(f"Random 5-fold CV WMAE (comparison): {cv_wmae_mean:,.0f} +/- {cv_wmae_std:,.0f}")


Time-based holdout WMAE (train<2024-06-30 -> val=2024-06-30): 69,217
Random 5-fold CV WMAE (comparison): 70,998 +/- 1,049


**Вывод раздела 6**: CV WMAE заметно лучше sanity floor — модель
эффективно использует признаки за пределами регионального. Time-based
holdout сравнивается со случайным K-Fold: если time-based WMAE заметно хуже,
это подтверждает temporal shift из ноутбука 1 и означает, что реальное
качество на приватном тесте, скорее всего, ближе к time-based оценке, чем
к случайному CV.

## 7. Интерпретация

In [13]:
# Финальная модель: фиксируем число итераций по средней best_iteration
# из CV, обучаем на ВСЕХ данных train без early stopping (иначе early
# stopping требовал бы утечки за счёт eval на части train).
final_iterations = int(np.mean(fold_best_iters)) + 1
print(f"final_iterations (avg best_iteration across folds + 1): {final_iterations}")

full_stats, full_fallback = fit_full_region_encoding(train, y, w, region_col=REGION_COL, min_count=MIN_GROUP_COUNT)
train["region_income_encoding"] = apply_region_encoding(train, full_stats, full_fallback, region_col=REGION_COL)
X_train_final = build_feature_matrix(train)

final_model = LogTargetCatBoost(
    cat_features=CAT_COLS,
    params={"iterations": final_iterations, "early_stopping_rounds": None},
)
final_model.fit(X_train_final, y, sample_weight=w)
print("final model trained on 100% of train.")


final_iterations (avg best_iteration across folds + 1): 1996


final model trained on 100% of train.


In [14]:
feature_importance = final_model.get_feature_importance()
feature_importance.head(20).to_frame("importance").to_csv(PARTIAL_OUTPUTS_DIR / "feature_importance.csv")
region_rank = list(feature_importance.index).index("region_income_encoding") + 1
print(f"region_income_encoding rank: {region_rank} / {len(feature_importance)}")
feature_importance.head(20)


region_income_encoding rank: 21 / 222


salary_6to12m_avg                                                                   13.905240
turn_cur_cr_avg_act_v2                                                               8.771262
hdb_bki_total_max_limit                                                              5.449615
gender                                                                               4.318767
incomeValue                                                                          3.238953
turn_cur_db_avg_act_v2                                                               3.190998
hdb_bki_total_cc_max_limit                                                           2.944515
dp_ils_paymentssum_avg_12m                                                           2.915312
hdb_bki_total_pil_max_limit                                                          2.861946
turn_cur_cr_avg_v2                                                                   2.134732
avg_cur_cr_turn                                             

**Вывод раздела 7**: `outputs/feature_importance.csv` сохранён (топ-20).
Место `region_income_encoding` печатается явно выше — сравнить с рангом
"зарплатных" признаков (`salary_6to12m_avg`,
`dp_payoutincomedata_payout_avg_3_month` и т.п.) из раздела 0.5/0.6 ТЗ:
региональный признак — легитимный, но умеренный по силе сигнал, как и
предсказывал раздел 0.6 (region-only объясняет лишь часть дисперсии).

## 8. Сабмит

In [ ]:
test["region_income_encoding"] = apply_region_encoding(test, full_stats, full_fallback, region_col=REGION_COL)
X_test_final = build_feature_matrix(test)

test_preds = final_model.predict(X_test_final)

submission = pd.DataFrame({ID_COL: test[ID_COL], "predict": test_preds})
submission.to_csv(OUTPUTS_DIR / "submission_baseline.csv", index=False, sep=";", decimal=",")
print("saved:", OUTPUTS_DIR / "submission_baseline.csv")
submission.head()


In [ ]:
# Sanity-check перед отправкой
assert submission["predict"].isna().sum() == 0, "NaN predictions found"
assert (submission["predict"] >= 0).all(), "negative predictions found"
assert len(submission) == len(test), "row count mismatch vs test.csv"

train_target_range = (train[TARGET_COL].min(), train[TARGET_COL].max())
pred_range = (submission["predict"].min(), submission["predict"].max())
print("train target range:", train_target_range)
print("prediction range:  ", pred_range)
print("all sanity checks passed.")


**Допущение (нужно подтвердить у организаторов)**: `sample_submission.csv`
в данных не обнаружен, поэтому формат сабмита выбран как `id;predict`
(разделитель `;`, запятая как десятичный разделитель — тот же формат, что
и во входных `train.csv`/`test.csv`) — см. README.md.

## Итоговая сводка (ноутбук 2)

- CV WMAE (5-fold, случайный сплит): печатается в разделе 6 выше, сравнение
  со sanity floor (региональный weighted-median baseline, раздел 5).
- Time-based holdout (последний месяц train) даёт независимую оценку под
  temporal shift, обнаруженный в ноутбуке 1.
- Финальная модель обучена на 100% train с региональным OOF-признаком,
  число итераций зафиксировано по средней `best_iteration` из CV (без
  дополнительного тюнинга, без ансамблей — см. раздел 6 ТЗ).
- `outputs/submission_baseline.csv`, `outputs/feature_importance.csv` и
  `outputs/feature_stats.csv` (из ноутбука 1) сохранены для дальнейшей
  работы.
